# RoadCheck — Обучение YOLOv8 на датасете RDD2022

Этот notebook обучает модель YOLOv8 для детекции дефектов дорожного покрытия.

**Датасет:** RDD2022 (Road Damage Detection 2022)  
**Классы:**
- `D00` — Продольные трещины (Longitudinal Crack)
- `D10` — Поперечные трещины (Transverse Crack)
- `D20` — Аллигаторные трещины (Alligator Crack)
- `D40` — Ямы / Выбоины (Pothole)

**Результат:** файл `best.pt` для использования в бэкенде RoadCheck.

## 1. Установка зависимостей

In [ ]:
!pip install ultralytics opencv-python-headless matplotlib tqdm lxml pyyaml

## 2. Импорты и настройки

In [ ]:
import shutil
import random
import xml.etree.ElementTree as ET
from pathlib import Path
from collections import Counter

import cv2
import yaml
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from tqdm import tqdm

# Воспроизводимость
SEED = 42
random.seed(SEED)

# Пути
BASE_DIR = Path.cwd()
RAW_DIR = BASE_DIR / "rdd2022_raw"          # Скачанный датасет
DATASET_DIR = BASE_DIR / "dataset_yolo"      # Сконвертированный в YOLO-формат
WEIGHTS_DST = BASE_DIR.parent / "app" / "ml" / "weights"  # Куда положить итоговые веса

# Классы RDD2022 → индексы YOLO
CLASS_MAP = {
    "D00": 0,  # Longitudinal Crack
    "D10": 1,  # Transverse Crack
    "D20": 2,  # Alligator Crack
    "D40": 3,  # Pothole
}
CLASS_NAMES = ["D00", "D10", "D20", "D40"]

print(f"Base dir:    {BASE_DIR}")
print(f"Raw dir:     {RAW_DIR}")
print(f"Dataset dir: {DATASET_DIR}")
print(f"Weights dst: {WEIGHTS_DST}")

## 3. Загрузка датасета RDD2022

RDD2022 доступен на [GitHub](https://github.com/sekilab/RoadDamageDetector).

Скачайте архив и распакуйте в папку `rdd2022_raw/` так, чтобы структура была:
```
rdd2022_raw/
├── Japan/
│   ├── train/
│   │   ├── images/
│   │   └── annotations/
│   │       └── xmls/
├── India/
├── Czech/
├── Norway/
├── United_States/
└── China_MotorBike/
```

Или запустите ячейку ниже для автоматического скачивания (требуется `gdown` или ручная загрузка).

In [ ]:
# Проверяем, есть ли уже скачанные данные
if RAW_DIR.exists() and any(RAW_DIR.iterdir()):
    countries = [d.name for d in RAW_DIR.iterdir() if d.is_dir()]
    print(f"Датасет найден! Страны: {countries}")
else:
    print("Датасет не найден.")
    print("")
    print("Скачайте RDD2022 вручную:")
    print("1. Перейдите: https://github.com/sekilab/RoadDamageDetector")
    print("2. Скачайте архив RDD2022")
    print(f"3. Распакуйте в: {RAW_DIR}")
    print("")
    print("Или используйте команду ниже (если данные доступны по прямой ссылке):")
    print("!pip install gdown && gdown <google_drive_id> -O rdd2022.zip && unzip rdd2022.zip -d rdd2022_raw")

## 4. Парсинг XML-аннотаций и конвертация в YOLO-формат

In [ ]:
def parse_voc_xml(xml_path: Path) -> list[dict]:
    """Парсит Pascal VOC XML и возвращает список объектов."""
    tree = ET.parse(xml_path)
    root = tree.getroot()

    size = root.find("size")
    img_w = int(size.find("width").text)
    img_h = int(size.find("height").text)

    objects = []
    for obj in root.findall("object"):
        name = obj.find("name").text.strip()
        if name not in CLASS_MAP:
            continue

        bbox = obj.find("bndbox")
        xmin = max(0, int(bbox.find("xmin").text))
        ymin = max(0, int(bbox.find("ymin").text))
        xmax = min(img_w, int(bbox.find("xmax").text))
        ymax = min(img_h, int(bbox.find("ymax").text))

        if xmax <= xmin or ymax <= ymin:
            continue

        # Конвертация в YOLO: нормализованные cx, cy, w, h
        cx = ((xmin + xmax) / 2) / img_w
        cy = ((ymin + ymax) / 2) / img_h
        w = (xmax - xmin) / img_w
        h = (ymax - ymin) / img_h

        objects.append({
            "class_id": CLASS_MAP[name],
            "class_name": name,
            "cx": cx, "cy": cy, "w": w, "h": h,
        })

    return objects


def objects_to_yolo_txt(objects: list[dict]) -> str:
    """Конвертирует список объектов в строку YOLO-формата."""
    lines = []
    for obj in objects:
        lines.append(f"{obj['class_id']} {obj['cx']:.6f} {obj['cy']:.6f} {obj['w']:.6f} {obj['h']:.6f}")
    return "\n".join(lines)


print("Функции парсинга готовы.")

In [ ]:
def collect_all_samples(raw_dir: Path) -> list[dict]:
    """
    Сканирует все страны в raw_dir и собирает пары (image, xml).
    Возвращает список словарей с путями.
    """
    samples = []
    
    for country_dir in sorted(raw_dir.iterdir()):
        if not country_dir.is_dir():
            continue
        
        train_dir = country_dir / "train"
        if not train_dir.exists():
            continue
            
        images_dir = train_dir / "images"
        xmls_dir = train_dir / "annotations" / "xmls"
        
        if not images_dir.exists() or not xmls_dir.exists():
            continue
        
        country = country_dir.name
        for img_path in sorted(images_dir.glob("*.jpg")):
            xml_path = xmls_dir / f"{img_path.stem}.xml"
            if xml_path.exists():
                samples.append({
                    "image": img_path,
                    "xml": xml_path,
                    "country": country,
                })
    
    print(f"Найдено {len(samples)} пар (image + xml)")
    
    # Статистика по странам
    country_counts = Counter(s["country"] for s in samples)
    for country, count in sorted(country_counts.items()):
        print(f"  {country}: {count} изображений")
    
    return samples


samples = collect_all_samples(RAW_DIR)

In [ ]:
# Статистика по классам
class_counter = Counter()
empty_count = 0

for sample in tqdm(samples, desc="Подсчёт аннотаций"):
    objects = parse_voc_xml(sample["xml"])
    if not objects:
        empty_count += 1
    for obj in objects:
        class_counter[obj["class_name"]] += 1

print(f"\nРаспределение классов:")
for cls_name in CLASS_NAMES:
    print(f"  {cls_name}: {class_counter.get(cls_name, 0)}")
print(f"  Пустых изображений (без дефектов): {empty_count}")
print(f"  Всего объектов: {sum(class_counter.values())}")

# Визуализация
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(class_counter.keys(), class_counter.values(), color=["#2196F3", "#4CAF50", "#FF9800", "#F44336"])
ax.set_title("Распределение классов в RDD2022")
ax.set_ylabel("Количество объектов")
plt.tight_layout()
plt.show()

## 5. Разделение на train/val и создание YOLO-структуры

In [ ]:
VAL_RATIO = 0.15  # 15% на валидацию


def build_yolo_dataset(
    samples: list[dict],
    output_dir: Path,
    val_ratio: float = VAL_RATIO,
) -> dict:
    """Конвертирует собранные сэмплы в YOLO-структуру с train/val split."""
    
    # Перемешиваем
    shuffled = samples.copy()
    random.shuffle(shuffled)
    
    val_count = int(len(shuffled) * val_ratio)
    val_samples = shuffled[:val_count]
    train_samples = shuffled[val_count:]
    
    stats = {"train": 0, "val": 0, "skipped": 0}
    
    for split_name, split_samples in [("train", train_samples), ("val", val_samples)]:
        img_dir = output_dir / "images" / split_name
        lbl_dir = output_dir / "labels" / split_name
        img_dir.mkdir(parents=True, exist_ok=True)
        lbl_dir.mkdir(parents=True, exist_ok=True)
        
        for sample in tqdm(split_samples, desc=f"Создание {split_name}"):
            objects = parse_voc_xml(sample["xml"])
            
            # Пропускаем пустые — нет дефектов
            if not objects:
                stats["skipped"] += 1
                continue
            
            # Уникальное имя: country_filename
            unique_name = f"{sample['country']}_{sample['image'].stem}"
            
            # Копируем изображение
            dst_img = img_dir / f"{unique_name}.jpg"
            shutil.copy2(sample["image"], dst_img)
            
            # Сохраняем лейбл
            dst_lbl = lbl_dir / f"{unique_name}.txt"
            dst_lbl.write_text(objects_to_yolo_txt(objects))
            
            stats[split_name] += 1
    
    return stats


# Удаляем старый датасет если есть
if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)

stats = build_yolo_dataset(samples, DATASET_DIR)
print(f"\nГотово!")
print(f"  Train: {stats['train']} изображений")
print(f"  Val:   {stats['val']} изображений")
print(f"  Пропущено (пустых): {stats['skipped']}")

In [ ]:
# Создаём data.yaml для YOLO
data_yaml = {
    "path": str(DATASET_DIR.resolve()),
    "train": "images/train",
    "val": "images/val",
    "nc": len(CLASS_NAMES),
    "names": CLASS_NAMES,
}

data_yaml_path = DATASET_DIR / "data.yaml"
with open(data_yaml_path, "w") as f:
    yaml.dump(data_yaml, f, default_flow_style=False, allow_unicode=True)

print("data.yaml создан:")
print(data_yaml_path.read_text())

## 6. Визуализация примеров из датасета

In [ ]:
COLORS = {0: "blue", 1: "green", 2: "orange", 3: "red"}


def show_sample(img_path: Path, lbl_path: Path, ax):
    """Отображает изображение с bounding boxes."""
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    ax.imshow(img)
    ax.set_title(img_path.stem, fontsize=8)
    ax.axis("off")

    if lbl_path.exists():
        for line in lbl_path.read_text().strip().split("\n"):
            parts = line.split()
            cls_id = int(parts[0])
            cx, cy, bw, bh = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])

            x1 = (cx - bw / 2) * w
            y1 = (cy - bh / 2) * h
            box_w = bw * w
            box_h = bh * h

            rect = patches.Rectangle(
                (x1, y1), box_w, box_h,
                linewidth=2, edgecolor=COLORS[cls_id], facecolor="none"
            )
            ax.add_patch(rect)
            ax.text(x1, y1 - 4, CLASS_NAMES[cls_id], color=COLORS[cls_id],
                    fontsize=7, fontweight="bold",
                    bbox=dict(boxstyle="round,pad=0.2", facecolor="white", alpha=0.7))


# Показать 8 случайных примеров из train
train_images = sorted((DATASET_DIR / "images" / "train").glob("*.jpg"))
sample_imgs = random.sample(train_images, min(8, len(train_images)))

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, img_path in zip(axes.flatten(), sample_imgs):
    lbl_path = DATASET_DIR / "labels" / "train" / f"{img_path.stem}.txt"
    show_sample(img_path, lbl_path, ax)

plt.suptitle("Примеры из обучающей выборки", fontsize=14)
plt.tight_layout()
plt.show()

## 7. Обучение YOLOv8

Используем **YOLOv8n** (nano) — быстрая и лёгкая модель. 
Для лучшего качества можно заменить на `yolov8s.pt` (small) или `yolov8m.pt` (medium).

In [ ]:
from ultralytics import YOLO

# Параметры обучения
MODEL_SIZE = "yolov8n.pt"   # nano (быстро). Альтернативы: yolov8s.pt, yolov8m.pt
EPOCHS = 50                  # Количество эпох (50-100 обычно достаточно)
IMG_SIZE = 640               # Размер входного изображения
BATCH_SIZE = 16              # Уменьшить если не хватает VRAM (8 или 4)
DEVICE = "0"                 # GPU. Поставьте "cpu" если нет GPU

print(f"Модель:    {MODEL_SIZE}")
print(f"Эпохи:    {EPOCHS}")
print(f"Img size: {IMG_SIZE}")
print(f"Batch:    {BATCH_SIZE}")
print(f"Device:   {DEVICE}")

In [ ]:
# Загружаем предобученную модель и запускаем обучение
model = YOLO(MODEL_SIZE)

results = model.train(
    data=str(data_yaml_path),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    project=str(BASE_DIR / "runs"),
    name="roadcheck",
    exist_ok=True,
    # Аугментации
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
    # Оптимизатор
    optimizer="auto",
    lr0=0.01,
    lrf=0.01,
    # Прочее
    patience=10,          # Early stopping
    save=True,
    plots=True,
    verbose=True,
)

print("\nОбучение завершено!")

## 8. Результаты обучения

In [ ]:
# Путь к результатам
run_dir = BASE_DIR / "runs" / "roadcheck"

# Показываем графики обучения
results_img = run_dir / "results.png"
if results_img.exists():
    img = cv2.imread(str(results_img))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(16, 8))
    plt.imshow(img)
    plt.axis("off")
    plt.title("Кривые обучения")
    plt.show()
else:
    print("results.png не найден")

In [ ]:
# Confusion matrix
cm_img = run_dir / "confusion_matrix_normalized.png"
if cm_img.exists():
    img = cv2.imread(str(cm_img))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(8, 8))
    plt.imshow(img)
    plt.axis("off")
    plt.title("Confusion Matrix (normalized)")
    plt.show()
else:
    print("confusion_matrix_normalized.png не найден")

## 9. Валидация лучшей модели

In [ ]:
# Загружаем лучшую модель
best_pt = run_dir / "weights" / "best.pt"
print(f"Лучшие веса: {best_pt}")
print(f"Файл существует: {best_pt.exists()}")

if best_pt.exists():
    best_model = YOLO(str(best_pt))
    metrics = best_model.val(
        data=str(data_yaml_path),
        imgsz=IMG_SIZE,
        device=DEVICE,
    )
    
    print(f"\n{'='*40}")
    print(f"mAP50:     {metrics.box.map50:.4f}")
    print(f"mAP50-95:  {metrics.box.map:.4f}")
    print(f"Precision: {metrics.box.mp:.4f}")
    print(f"Recall:    {metrics.box.mr:.4f}")
    print(f"{'='*40}")
    
    # Per-class метрики
    print("\nПо классам:")
    for i, name in enumerate(CLASS_NAMES):
        print(f"  {name}: mAP50={metrics.box.ap50[i]:.4f}  mAP50-95={metrics.box.ap[i]:.4f}")
else:
    print("best.pt не найден. Сначала запустите обучение.")

## 10. Тестовые предсказания

In [ ]:
if best_pt.exists():
    best_model = YOLO(str(best_pt))
    
    # Берём несколько изображений из валидации
    val_images = sorted((DATASET_DIR / "images" / "val").glob("*.jpg"))
    test_imgs = random.sample(val_images, min(6, len(val_images)))
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    
    for ax, img_path in zip(axes.flatten(), test_imgs):
        results = best_model(str(img_path), verbose=False)[0]
        
        # Рисуем результат
        annotated = results.plot()
        annotated = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
        
        ax.imshow(annotated)
        ax.set_title(f"{img_path.stem} — {len(results.boxes)} дефект(ов)", fontsize=9)
        ax.axis("off")
    
    plt.suptitle("Предсказания модели на валидации", fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print("best.pt не найден.")

## 11. Копирование весов в проект

In [ ]:
if best_pt.exists():
    WEIGHTS_DST.mkdir(parents=True, exist_ok=True)
    dst = WEIGHTS_DST / "best.pt"
    shutil.copy2(best_pt, dst)
    
    size_mb = dst.stat().st_size / (1024 * 1024)
    print(f"Веса скопированы: {dst}")
    print(f"Размер: {size_mb:.1f} MB")
    print(f"")
    print(f"Теперь в .env установите USE_MOCK_ML=false")
    print(f"и перезапустите бэкенд — модель будет работать!")
else:
    print("best.pt не найден. Сначала обучите модель.")

## 12. Информация о модели

In [ ]:
if best_pt.exists():
    info_model = YOLO(str(best_pt))
    
    print("Классы модели:")
    for idx, name in info_model.names.items():
        print(f"  {idx}: {name}")
    
    print(f"\nКоличество параметров: {sum(p.numel() for p in info_model.model.parameters()):,}")
    print(f"Архитектура: YOLOv8n")
    print(f"Input size: {IMG_SIZE}x{IMG_SIZE}")
else:
    print("best.pt не найден.")